# PharmaLens AI — Data Intelligence Layer
**Central reusable ingestion, normalization, source detection, target detection, quality and provenance engine.**

Principle: **Minimum Data In → Maximum Intelligence Out.**

This notebook is the reference implementation. In production, Gemini should move reusable logic into the existing project architecture instead of duplicating it in every notebook.


In [ ]:
from pathlib import Path
import re, hashlib
import pandas as pd
import numpy as np

INPUT_DIR = Path("data/incoming")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED = {".xlsx", ".xls", ".csv", ".parquet"}


## Canonical field aliases
Flexible mapping supports common pharmaceutical sales schemas without forcing users to rename columns.


In [ ]:
ALIASES = {
    "product": ["product","product name","brand","brand name","sku","item","item name","medicine","drug"],
    "units": ["units","sales units","quantity","qty","volume","packs","sales qty"],
    "sales_value": ["sales value","sales_value","value","sales","revenue","net sales","sales amount"],
    "unit_price": ["unit price","price","selling price","sales price","unit_price"],
    "date": ["date","transaction date","invoice date"],
    "month": ["month","period","sales month"],
    "year": ["year","sales year"],
    "target_units": ["target units","unit target","target qty","target quantity","target_units"],
    "target_value": ["target value","value target","sales target","target sales","target_value"],
    "source": ["source","sales source","data source"],
    "channel": ["channel","sales channel","distribution channel"],
    "distributor": ["distributor","supplier","wholesaler","distribution partner"],
    "territory": ["territory","region","area","geography"],
    "rep": ["rep","sales rep","representative","employee","owner"],
    "customer": ["customer","account","hospital","pharmacy","customer name"]
}
def norm_col(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x).strip().lower()).strip()
def detect_schema(df):
    normalized = {norm_col(c):c for c in df.columns}
    mapping, confidence = {}, {}
    for canonical, candidates in ALIASES.items():
        for candidate in candidates:
            if norm_col(candidate) in normalized:
                mapping[canonical] = normalized[norm_col(candidate)]
                confidence[canonical] = 1.0
                break
    return mapping, confidence


## Source detection
Priority: explicit field → sheet name → filename → `Other`.
Never invent a distributor/source. Low confidence should become a one-click frontend confirmation.


In [ ]:
SOURCE_HINTS = {
    "PM/Retail": ["pm","retail","pharmacy","community"],
    "AM/Hospital": ["am","hospital","institutional","institution"],
    "UCP": ["ucp","المتحدة"],
    "Ibn Sina": ["ibn sina","ابن سينا"],
    "Egy Drug": ["egy drug","egyptian drug","المصرية"],
    "Aramco": ["aramco"]
}
def infer_source(filename="", sheet_name="", explicit=None):
    if explicit:
        return {"source":str(explicit),"type":"Explicit","confidence":1.0}
    text = f"{filename} {sheet_name}".lower()
    for entity,hints in SOURCE_HINTS.items():
        if any(h in text for h in hints):
            typ = "Channel" if entity in ("PM/Retail","AM/Hospital") else "Distributor"
            return {"source":entity,"type":typ,"confidence":0.94}
    return {"source":"Other","type":"Other","confidence":0.40}


## Ingestion
Supports one file, multiple files, one workbook with multiple sheets, sales-only, target-only, and sales+target in one sheet.


In [ ]:
def read_one(path):
    path = Path(path)
    if path.suffix.lower() in {".xlsx",".xls"}:
        return pd.read_excel(path, sheet_name=None)
    if path.suffix.lower() == ".csv":
        return {path.stem: pd.read_csv(path)}
    if path.suffix.lower() == ".parquet":
        return {path.stem: pd.read_parquet(path)}
    raise ValueError(f"Unsupported file type: {path.suffix}")

def load_inputs(paths):
    records=[]
    for p in paths:
        for sheet,df in read_one(p).items():
            records.append({"file":str(p),"sheet":str(sheet),"data":df.copy()})
    return records


In [ ]:
def canonicalize(df, filename="", sheet_name=""):
    df=df.copy()
    mapping, confidence=detect_schema(df)
    out=df.copy()
    for canonical,original in mapping.items():
        out[canonical]=df[original]
    explicit = out["source"].iloc[0] if "source" in out.columns and len(out) else None
    src=infer_source(filename,sheet_name,explicit)
    out["detected_source"]=src["source"]
    out["source_type"]=src["type"]
    out["source_confidence"]=src["confidence"]
    out["source_file"]=str(filename)
    out["source_sheet"]=str(sheet_name)
    return out,mapping,confidence


## Sales value + time normalization
If Sales Value is missing and Units + Unit Price exist, calculate `Sales Value = Units × Unit Price`.
Never fabricate a price or sales value.


In [ ]:
def prepare_sales(df):
    df=df.copy()
    for c in ["units","unit_price","sales_value"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c],errors="coerce")
    if "sales_value" not in df.columns and {"units","unit_price"}.issubset(df.columns):
        df["sales_value"]=df["units"]*df["unit_price"]
        df["sales_value_calculated"]=True
    elif "sales_value" in df.columns:
        df["sales_value_calculated"]=False
        if {"units","unit_price"}.issubset(df.columns):
            mask=df["sales_value"].isna() & df["units"].notna() & df["unit_price"].notna()
            df.loc[mask,"sales_value"]=df.loc[mask,"units"]*df.loc[mask,"unit_price"]
            df.loc[mask,"sales_value_calculated"]=True
    if "date" in df.columns:
        d=pd.to_datetime(df["date"],errors="coerce")
        df["date"]=d; df["year"]=d.dt.year; df["month"]=d.dt.month
    elif "month" in df.columns:
        parsed=pd.to_datetime(df["month"],errors="coerce")
        df["month"]=parsed.dt.month.fillna(pd.to_numeric(df["month"],errors="coerce"))
        if "year" in df.columns:
            df["year"]=pd.to_numeric(df["year"],errors="coerce")
    return df


## Target detection
`NO_TARGET` is a valid state. Missing targets are never treated as zero.


In [ ]:
def target_status(df):
    hv="target_value" in df.columns and df["target_value"].notna().any()
    hu="target_units" in df.columns and df["target_units"].notna().any()
    if not (hv or hu): return "NO_TARGET"
    if hv and hu: return "TARGET_AVAILABLE"
    return "TARGET_PARTIAL"

def quality_report(df):
    return {
        "rows":len(df),
        "columns":len(df.columns),
        "products":int(df["product"].nunique(dropna=True)) if "product" in df.columns else None,
        "has_sales_value":"sales_value" in df.columns and df["sales_value"].notna().any(),
        "target_status":target_status(df),
        "source_count":int(df["detected_source"].nunique()) if "detected_source" in df.columns else 0,
        "missing_counts":{c:int(df[c].isna().sum()) for c in df.columns if df[c].isna().any()}
    }


## Unified data contract
Production contract should expose logical datasets:
`sales`, `targets`, `dimensions`, `data_quality`, `provenance`, `source_registry`, `periods`.


In [ ]:
def build_unified_dataset(paths):
    frames=[]; metadata=[]
    for rec in load_inputs(paths):
        c,mapping,confidence=canonicalize(rec["data"],rec["file"],rec["sheet"])
        c=prepare_sales(c)
        frames.append(c)
        metadata.append({
            "file":rec["file"],"sheet":rec["sheet"],"mapping":mapping,
            "mapping_confidence":confidence,"source":c["detected_source"].iloc[0] if len(c) else "Other",
            "target_status":target_status(c),"quality":quality_report(c)
        })
    unified=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame()
    return unified,metadata

def split_contract(unified):
    sales_cols=[c for c in ["product","units","sales_value","unit_price","date","month","year",
                            "detected_source","source_type","source_file","source_sheet",
                            "channel","distributor","territory","rep","customer"] if c in unified.columns]
    target_cols=[c for c in ["product","target_units","target_value","month","year",
                             "detected_source","source_file","source_sheet"] if c in unified.columns]
    return {"sales":unified[sales_cols].copy() if sales_cols else pd.DataFrame(),
            "targets":unified[target_cols].copy() if target_cols else pd.DataFrame(),
            "all_data":unified}


## Production integration
Move reusable functions into the existing project package (for example `pharmalens/data_layer/`) and make all notebooks import the shared layer. Do not duplicate ingestion/normalization logic in each notebook.
